```
fuss/
├── features.pkl   -- {sample_id: {'audio_path': <path_relative_to_dataset_root>}}
├── targets.pkl    -- {sample_id: {'source_audio_paths': <tuple[str, ...]>, 'num_sources': <int>}}
├── train_keys.pkl -- {index: {'key': <sample_id>}}
├── val_keys.pkl   -- {index: {'key': <sample_id>}}
├── test_keys.pkl  -- {index: {'key': <sample_id>}}
└── manifest.csv   -- sample-level manifest with split, mixture path, and source paths
```

In [1]:
import json
import pickle
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import display
from tqdm.auto import tqdm


/Users/nsborodin/repos/autodition/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
repo_root = Path('.').resolve().parent
data_dir = repo_root / 'data'
raw_dir = data_dir / 'raw' / 'FUSS'
preprocessed_dir = data_dir / 'preprocessed' / 'fuss'
download_dir = raw_dir / '_downloads'

dataset_variant = 'ssdata_reverb'  # 'ssdata' or 'ssdata_reverb'
download_from_zenodo = True
extract_archives = True
force_overwrite = True
consistency_sample_size = 128

zenodo_record_id = '4012661'
archive_names = {
    'ssdata': 'FUSS_ssdata.tar.gz',
    'ssdata_reverb': 'FUSS_ssdata_reverb.tar.gz',
}
archive_name = archive_names[dataset_variant]
download_urls = {
    archive_name: f'https://zenodo.org/records/{zenodo_record_id}/files/{archive_name}?download=1',
    'FUSS_license_doc.tar.gz': f'https://zenodo.org/records/{zenodo_record_id}/files/FUSS_license_doc.tar.gz?download=1',
}
split_map = {'train': 'train', 'validation': 'val', 'eval': 'test'}
dataset_root = raw_dir / dataset_variant

print(f'repo_root:         {repo_root}')
print(f'raw_dir:           {raw_dir}')
print(f'dataset_variant:   {dataset_variant}')
print(f'dataset_root:      {dataset_root}')
print(f'preprocessed_dir:  {preprocessed_dir}')


repo_root:         /Users/nsborodin/repos/autodition
raw_dir:           /Users/nsborodin/repos/autodition/data/raw/FUSS
dataset_variant:   ssdata_reverb
dataset_root:      /Users/nsborodin/repos/autodition/data/raw/FUSS/ssdata_reverb
preprocessed_dir:  /Users/nsborodin/repos/autodition/data/preprocessed/fuss


## Download

In [4]:
raw_dir.mkdir(parents=True, exist_ok=True)
download_dir.mkdir(parents=True, exist_ok=True)

if download_from_zenodo:
    for filename, url in download_urls.items():
        target_path = download_dir / filename
        if target_path.exists() and not force_overwrite:
            print(f'Skipping existing file: {target_path.name}')
            continue

        print(f'Downloading {filename}...')
        subprocess.run(
            [
                'curl',
                '-L',
                '--fail',
                '--retry',
                '3',
                '--continue-at',
                '-',
                url,
                '-o',
                str(target_path),
            ],
            check=True,
        )
else:
    print('Skipping Zenodo download. Set download_from_zenodo=True to enable it.')
    print('Expected archives:')
    for filename in download_urls:
        print(f'  - {download_dir / filename}')


** Resuming transfer from byte position 238542848
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 7297M  100 7297M    0     0  14.8M      0  0:08:12  0:08:12 --:--:-- 13.1M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

100  4022  100  4022    0     0  13809      0 --:--:-- --:--:-- --:--:-- 13773


In [5]:
archives = [download_dir / filename for filename in download_urls if (download_dir / filename).exists()]
print(f'Found {len(archives)} archive(s) in {download_dir}')
for archive_path in archives:
    marker_path = raw_dir / f'.{archive_path.name}.extracted'
    if not extract_archives:
        print(f'Skipping extraction for {archive_path.name}')
        continue
    if marker_path.exists() and not force_overwrite:
        print(f'Already extracted: {archive_path.name}')
        continue

    print(f'Extracting {archive_path.name}...')
    subprocess.run(['tar', '-xzf', str(archive_path), '-C', str(raw_dir)], check=True)
    marker_path.write_text('ok\n')

if not dataset_root.exists():
    matches = sorted(path for path in raw_dir.rglob(dataset_variant) if path.is_dir())
    if len(matches) == 1:
        dataset_root = matches[0]
        print(f'Using discovered dataset root: {dataset_root}')
    else:
        print(f'Dataset root not found yet: {dataset_root}')


Found 2 archive(s) in /Users/nsborodin/repos/autodition/data/raw/FUSS/_downloads
Extracting FUSS_ssdata_reverb.tar.gz...
Extracting FUSS_license_doc.tar.gz...


## Inspect

In [6]:
assert dataset_root.exists(), f'Dataset root does not exist: {dataset_root}'

split_summary_rows = []
for official_split in split_map:
    split_dir = dataset_root / official_split
    jam_files = sorted(split_dir.rglob('*.jams')) if split_dir.exists() else []
    wav_files = sorted(split_dir.rglob('*.wav')) if split_dir.exists() else []
    split_summary_rows.append(
        {
            'official_split': official_split,
            'repo_split': split_map[official_split],
            'exists': split_dir.exists(),
            'jams_files': len(jam_files),
            'wav_files': len(wav_files),
            'path': split_dir.as_posix(),
        }
    )

display(pd.DataFrame(split_summary_rows))


,official_split,repo_split,exists,jams_files,wav_files,path
0,train,train,True,20000,70067,/Users/nsborodin/repos/autodition/data/raw/FUS...
1,validation,val,True,1000,3516,/Users/nsborodin/repos/autodition/data/raw/FUS...
2,eval,test,True,1000,3445,/Users/nsborodin/repos/autodition/data/raw/FUS...


In [7]:
example_jams = next(dataset_root.rglob('*.jams'), None)
assert example_jams is not None, f'No .jams files found under {dataset_root}'

example_payload = json.loads(example_jams.read_text())
scaper_annotations = [ann for ann in example_payload.get('annotations', []) if ann.get('namespace') == 'scaper']
assert scaper_annotations, f'No scaper annotation found in {example_jams}'

example_scaper = scaper_annotations[0].get('sandbox', {}).get('scaper', {})
print(f'Example JAMS: {example_jams.relative_to(raw_dir)}')
print('soundscape_audio_path:', example_scaper.get('soundscape_audio_path'))
print('isolated_events_audio_path count:', len(example_scaper.get('isolated_events_audio_path', [])))


Example JAMS: ssdata_reverb/train/example09610.jams
soundscape_audio_path: /data/DCASE2020/ssdata/train/example09610.wav
isolated_events_audio_path count: 3


## Build

In [8]:
def get_scaper_annotation(payload: dict) -> dict:
    for annotation in payload.get('annotations', []):
        if annotation.get('namespace') == 'scaper':
            return annotation
    raise ValueError('No scaper annotation found in JAMS payload')


def resolve_scaper_audio_path(path_str: str, split_dir: Path, item_base_name: str) -> Path:
    path_str = str(path_str)
    if item_base_name in path_str:
        suffix = path_str.split(item_base_name, 1)[1]
        candidate = split_dir / f'{item_base_name}{suffix}'
        if candidate.exists():
            return candidate

    direct_candidate = split_dir / Path(path_str).name
    if direct_candidate.exists():
        return direct_candidate

    recursive_candidates = sorted(
        path for path in split_dir.rglob('*') if path.is_file() and path.name == Path(path_str).name
    )
    if len(recursive_candidates) == 1:
        return recursive_candidates[0]

    stem_candidates = sorted(
        path for path in split_dir.rglob(f'{item_base_name}*') if path.is_file() and path.suffix.lower() == '.wav'
    )
    if len(stem_candidates) == 1:
        return stem_candidates[0]

    raise FileNotFoundError(
        f'Could not resolve audio path {path_str!r} for {split_dir / (item_base_name + ".jams")}'
    )


def build_sample_id(official_split: str, jams_path: Path) -> str:
    return f'{official_split}__{jams_path.stem}'


rows = []
for official_split, repo_split in split_map.items():
    split_dir = dataset_root / official_split
    jam_files = sorted(split_dir.rglob('*.jams'))
    for jams_path in tqdm(jam_files, desc=f'Parsing {official_split}'):
        payload = json.loads(jams_path.read_text())
        scaper = get_scaper_annotation(payload).get('sandbox', {}).get('scaper', {})
        mix_path = resolve_scaper_audio_path(scaper['soundscape_audio_path'], split_dir, jams_path.stem)
        source_paths = [
            resolve_scaper_audio_path(path_str, split_dir, jams_path.stem)
            for path_str in scaper.get('isolated_events_audio_path', [])
        ]

        rows.append(
            {
                'sample_id': build_sample_id(official_split, jams_path),
                'split': repo_split,
                'official_split': official_split,
                'jams_rel_path': jams_path.relative_to(raw_dir).as_posix(),
                'mixture_rel_path': mix_path.relative_to(raw_dir).as_posix(),
                'source_rel_paths': tuple(path.relative_to(raw_dir).as_posix() for path in source_paths),
                'num_sources': len(source_paths),
            }
        )

manifest = pd.DataFrame(rows).sort_values(['split', 'sample_id']).reset_index(drop=True)
print(f'Samples in manifest: {len(manifest)}')
manifest.head()


Parsing eval: 100%|██████████| 1000/1000 [00:00<00:00, 6604.11it/s]


Samples in manifest: 22000


,sample_id,split,official_split,jams_rel_path,mixture_rel_path,source_rel_paths,num_sources
0,eval__example0000,test,eval,ssdata_reverb/eval/example0000.jams,ssdata_reverb/eval/example0000.wav,(ssdata_reverb/eval/example0000_sources/backgr...,4
1,eval__example0001,test,eval,ssdata_reverb/eval/example0001.jams,ssdata_reverb/eval/example0001.wav,(ssdata_reverb/eval/example0001_sources/backgr...,1
2,eval__example0002,test,eval,ssdata_reverb/eval/example0002.jams,ssdata_reverb/eval/example0002.wav,(ssdata_reverb/eval/example0002_sources/backgr...,4
3,eval__example0003,test,eval,ssdata_reverb/eval/example0003.jams,ssdata_reverb/eval/example0003.wav,(ssdata_reverb/eval/example0003_sources/backgr...,2
4,eval__example0004,test,eval,ssdata_reverb/eval/example0004.jams,ssdata_reverb/eval/example0004.wav,(ssdata_reverb/eval/example0004_sources/backgr...,4


In [9]:
metadata_rows = []
for row in tqdm(manifest.itertuples(index=False), total=len(manifest), desc='Reading metadata'):
    mixture_info = sf.info(str(raw_dir / row.mixture_rel_path))
    source_infos = [sf.info(str(raw_dir / path)) for path in row.source_rel_paths]
    metadata_rows.append(
        {
            'sample_id': row.sample_id,
            'split': row.split,
            'official_split': row.official_split,
            'num_sources': row.num_sources,
            'mixture_sr': mixture_info.samplerate,
            'mixture_frames': mixture_info.frames,
            'mixture_duration_seconds': mixture_info.frames / mixture_info.samplerate,
            'source_srs_match': all(info.samplerate == mixture_info.samplerate for info in source_infos),
            'source_frames_match': all(info.frames == mixture_info.frames for info in source_infos),
        }
    )

metadata = pd.DataFrame(metadata_rows)

consistency_rows = []
sampled_manifest = manifest.sample(n=min(consistency_sample_size, len(manifest)), random_state=12345)
for row in tqdm(sampled_manifest.itertuples(index=False), total=len(sampled_manifest), desc='Checking consistency'):
    mixture, _ = sf.read(raw_dir / row.mixture_rel_path, dtype='float32')
    source_sum = np.zeros_like(mixture)
    for source_rel_path in row.source_rel_paths:
        source_audio, _ = sf.read(raw_dir / source_rel_path, dtype='float32')
        source_sum = source_sum + source_audio

    raw_l1 = float(np.abs(mixture - source_sum).mean())
    denominator = float((source_sum * source_sum).sum())
    alpha = float((mixture * source_sum).sum() / denominator) if denominator > 1e-12 else 0.0
    scaled_l1 = float(np.abs(mixture - alpha * source_sum).mean())
    cosine_denominator = float(np.linalg.norm(mixture) * np.linalg.norm(source_sum))
    cosine_similarity = float(np.dot(mixture, source_sum) / cosine_denominator) if cosine_denominator > 1e-12 else 0.0

    consistency_rows.append(
        {
            'sample_id': row.sample_id,
            'split': row.split,
            'raw_l1': raw_l1,
            'scaled_l1': scaled_l1,
            'alpha': alpha,
            'cosine_similarity': cosine_similarity,
        }
    )

consistency = pd.DataFrame(consistency_rows)


Checking consistency: 100%|██████████| 128/128 [00:00<00:00, 263.20it/s]


## EDA

In [10]:
print('Samples by split:')
display(manifest['split'].value_counts().rename_axis('split').to_frame('num_samples'))

print('Samples by official split:')
display(manifest['official_split'].value_counts().rename_axis('official_split').to_frame('num_samples'))

print('Number of sources by split:')
display(manifest.groupby('split')['num_sources'].value_counts().unstack(fill_value=0).sort_index(axis=1))

print('Audio metadata summary:')
display(metadata[['mixture_sr', 'mixture_duration_seconds', 'num_sources']].describe().T)

print('Sample-rate and frame consistency:')
display(
    pd.DataFrame(
        {
            'source_srs_match_rate': [metadata['source_srs_match'].mean()],
            'source_frames_match_rate': [metadata['source_frames_match'].mean()],
        }
    )
)

print('Mixture/source consistency summary (sampled subset):')
display(consistency[['raw_l1', 'scaled_l1', 'alpha', 'cosine_similarity']].describe().T)

print('Worst consistency examples in the sampled subset:')
display(consistency.sort_values('raw_l1', ascending=False).head(10))


Samples by split:


,num_samples
split,
train,20000
test,1000
val,1000


Samples by official split:


,num_samples
official_split,
train,20000
eval,1000
validation,1000


Number of sources by split:


...

,source_srs_match_rate,source_frames_match_rate
0,1.0,1.0


Mixture/source consistency summary (sampled subset):


,count,mean,std,min,25%,50%,75%,max
raw_l1,128.0,0.000005,0.000005,0.000000,1.028061e-07,0.000003,0.000009,0.000018
scaled_l1,128.0,0.000005,0.000005,0.000000,1.082561e-07,0.000003,0.000009,0.000019
alpha,128.0,1.000193,0.000357,0.999977,1.000000e+00,1.000066,1.000205,1.002416
cosine_similarity,128.0,0.999988,0.000028,0.999760,9.999876e-01,0.999999,1.000000,1.000002


Worst consistency examples in the sampled subset:


,sample_id,split,raw_l1,scaled_l1,alpha,cosine_similarity
127,eval__example0087,test,0.000018,0.000019,1.000202,0.999998
50,train__example11339,train,0.000017,0.000017,1.001301,0.999929
55,train__example08488,train,0.000016,0.000016,1.000211,0.999996
86,train__example07153,train,0.000016,0.000016,1.000102,0.999999
67,train__example10428,train,0.000016,0.000017,1.002001,0.999921
112,train__example19144,train,0.000016,0.000016,1.000039,0.999999
109,train__example05389,train,0.000016,0.000016,1.000257,1.000000
33,train__example13526,train,0.000016,0.000016,1.000213,0.999999
99,train__example10313,train,0.000015,0.000015,1.000081,0.999999
26,train__example16697,train,0.000015,0.000015,1.000061,1.000000


## Save

In [11]:
features = {}
targets = {}

for row in manifest.itertuples(index=False):
    features[row.sample_id] = {
        'audio_path': row.mixture_rel_path,
        'official_split': row.official_split,
        'jams_path': row.jams_rel_path,
    }
    targets[row.sample_id] = {
        'source_audio_paths': tuple(row.source_rel_paths),
        'num_sources': int(row.num_sources),
    }

train_ids = manifest.loc[manifest['split'] == 'train', 'sample_id'].tolist()
val_ids = manifest.loc[manifest['split'] == 'val', 'sample_id'].tolist()
test_ids = manifest.loc[manifest['split'] == 'test', 'sample_id'].tolist()

train_keys = {idx: {'key': sample_id} for idx, sample_id in enumerate(train_ids)}
val_keys = {idx: {'key': sample_id} for idx, sample_id in enumerate(val_ids)}
test_keys = {idx: {'key': sample_id} for idx, sample_id in enumerate(test_ids)}

preprocessed_dir.mkdir(parents=True, exist_ok=True)

artifacts = {
    preprocessed_dir / 'features.pkl': features,
    preprocessed_dir / 'targets.pkl': targets,
    preprocessed_dir / 'train_keys.pkl': train_keys,
    preprocessed_dir / 'val_keys.pkl': val_keys,
    preprocessed_dir / 'test_keys.pkl': test_keys,
}

for artifact_path, payload in artifacts.items():
    if artifact_path.exists() and not force_overwrite:
        raise FileExistsError(f'{artifact_path} already exists. Set force_overwrite=True to replace it.')
    with open(artifact_path, 'wb') as stream:
        pickle.dump(payload, stream)

manifest.to_csv(preprocessed_dir / 'manifest.csv', index=False)
metadata.to_csv(preprocessed_dir / 'audio_metadata.csv', index=False)
consistency.to_csv(preprocessed_dir / 'consistency_check.csv', index=False)

print(f'Saved features:    {len(features)} entries -> {preprocessed_dir / "features.pkl"}')
print(f'Saved targets:     {len(targets)} entries -> {preprocessed_dir / "targets.pkl"}')
print(f'Saved train_keys:  {len(train_keys):>5} entries -> {preprocessed_dir / "train_keys.pkl"}')
print(f'Saved val_keys:    {len(val_keys):>5} entries -> {preprocessed_dir / "val_keys.pkl"}')
print(f'Saved test_keys:   {len(test_keys):>5} entries -> {preprocessed_dir / "test_keys.pkl"}')
print(f'Saved manifest:    {len(manifest)} rows -> {preprocessed_dir / "manifest.csv"}')
print(f'Saved metadata:    {len(metadata)} rows -> {preprocessed_dir / "audio_metadata.csv"}')
print(f'Saved consistency: {len(consistency)} rows -> {preprocessed_dir / "consistency_check.csv"}')


Saved features:    22000 entries -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/features.pkl
Saved targets:     22000 entries -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/targets.pkl
Saved train_keys:  20000 entries -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/train_keys.pkl
Saved val_keys:     1000 entries -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/val_keys.pkl
Saved test_keys:    1000 entries -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/test_keys.pkl
Saved manifest:    22000 rows -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/manifest.csv
Saved metadata:    22000 rows -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/audio_metadata.csv
Saved consistency: 128 rows -> /Users/nsborodin/repos/autodition/data/preprocessed/fuss/consistency_check.csv
